# Lab 2.3 · A dbt project for Qafila

Day 2 · Session 3 (dbt: transformation as software) · 25 min. What you'll do:
- tour a real dbt project: sources, staging → intermediate → marts, macros, configs;
- write a **staging model** and an **incremental fact model** yourself, and build the DAG;
- watch an incremental run pick up a late change, and read the **lineage** dbt derives from `ref()`.

The complete project is in `labs/dbt_qafila/`. This lab works on a **copy** with two models removed
(`stg_orders` and `fct_order_items`), and on a copy of the warehouse, so nothing you break affects other labs.
dbt runs **in-process** from Python (`U.run_dbt([...])` wraps `dbtRunner`); in a terminal you'd type `dbt build` etc.

**Tasks**
- [ ] 1 · Tour the project and list its models (4')
- [ ] 2 · Write `stg_orders` and build the staging layer (6')
- [ ] 3 · Write the incremental `fct_order_items` and build the marts (7')
- [ ] 4 · Change a source row; watch the incremental run (5')
- [ ] 5 · Read the lineage from the manifest (3')
- [ ] ✱ Stretch · Add a weekly category mart and document it

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/data-engineering.git"
    if not Path("/content/data-engineering").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/data-engineering"], check=True)
    os.chdir("/content/data-engineering/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("de_utils.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import de_utils as U
U.ensure_data()
print("data:", U.RAW, "| quick mode:", U.QUICK)

data: /content/data-engineering/labs/data/raw | quick mode: False


In [2]:
import json
import shutil

import duckdb

U.build_warehouse()
OUT = U.lab_dir("D2_3", reset=True)
PROJ = OUT / "dbt_qafila"
DB = OUT / "qafila.duckdb"
shutil.copytree(U.DBT_PROJECT, PROJ, ignore=shutil.ignore_patterns("target", "logs", "dbt_packages"))
shutil.copy(U.WAREHOUSE_DB, DB)                                  # bronze layer only
for m in ["models/staging/stg_orders.sql", "models/marts/fct_order_items.sql"]:   # you'll write these two
    (PROJ / m).write_text("-- TODO (Lab 2.3): placeholder so the project still parses\nselect 1 as todo\n", encoding="utf-8")


def dbt(*args, quiet=False):
    """Run dbt on this lab's project and database."""
    return U.run_dbt(list(args), project_dir=PROJ, db_path=DB, quiet=quiet)


def sql(query):
    with duckdb.connect(str(DB)) as c:          # same settings as dbt's own connection in this process
        return c.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1 · Tour the project and list its models (4')

A dbt project is a folder of SQL `SELECT`s plus YAML. dbt turns each `SELECT` into a view or table, in dependency order.

In [3]:
for p in sorted(PROJ.rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJ))

dbt_project.yml
macros/generate_schema_name.sql
macros/text.sql
models/intermediate/int_delivery_timeline.sql
models/intermediate/int_payments_per_order.sql
models/marts/_marts.yml
models/marts/dim_customers.sql
models/marts/dim_date.sql
models/marts/dim_products.sql
models/marts/dim_stores.sql
models/marts/fct_order_items.sql
models/marts/fct_orders.sql
models/marts/mart_daily_city_sales.sql
models/quarantine/qrn_order_items.sql
models/staging/_sources.yml
models/staging/_staging.yml
models/staging/stg_customers.sql
models/staging/stg_delivery_events.sql
models/staging/stg_order_items.sql
models/staging/stg_orders.sql
models/staging/stg_payments.sql
models/staging/stg_products.sql
models/staging/stg_reviews.sql
models/staging/stg_stores.sql
models/staging/stg_supplier_prices.sql
profiles.yml
seeds/promo_codes.csv
tests/assert_order_totals_match_payments.sql
tests/generic/positive.sql


In [4]:
print((PROJ / "dbt_project.yml").read_text(encoding="utf-8"))
print((PROJ / "models/staging/stg_order_items.sql").read_text(encoding="utf-8"))

name: qafila
version: "1.0.0"
config-version: 2
profile: qafila

model-paths: ["models"]
macro-paths: ["macros"]
test-paths: ["tests"]
seed-paths: ["seeds"]
target-path: "target"
clean-targets: ["target", "dbt_packages"]

models:
  qafila:
    staging:
      +schema: staging
      +materialized: view
    intermediate:
      +schema: intermediate
      +materialized: view
    marts:
      +schema: marts
      +materialized: table
    quarantine:
      +schema: quarantine
      +materialized: table

seeds:
  qafila:
    +schema: reference

vars:
  sla_minutes: 30          # Qafila's delivery promise
  unknown_key: -1          # dimension member for missing / erased references

-- One row per order line. Negative quantities exist in the source (see the issue log);
-- they are kept here and handled by tests and the fact table.
select
    order_id,
    line_no,
    order_id * 100 + line_no                          as order_line_id,
    product_id,
    quantity,
    unit_price,
    discount,

Three things to notice:
- `{{ source('raw', 'order_items') }}` points at a declared **source** (`models/staging/_sources.yml`), never at a table name.
- Folder-level configs in `dbt_project.yml`: staging models are **views** in schema `staging`, marts are **tables** in `marts`.
- Reusable SQL lives in **macros** (`macros/text.sql`: Arabic normalization, number parsing, E.164 mobiles, Riyadh time).

`dbt ls` lists what dbt sees. `+fct_orders` means "fct_orders and everything upstream of it".

In [5]:
res = dbt("ls", "--resource-type", "model", "--output", "name", quiet=True)
print(sorted(res.result))

dim_customers
dim_date
dim_products
dim_stores
fct_order_items
fct_orders
int_delivery_timeline
int_payments_per_order
mart_daily_city_sales
qrn_order_items
stg_customers
stg_delivery_events
stg_order_items
stg_orders
stg_payments
stg_products
stg_reviews
stg_stores
stg_supplier_prices
['dim_customers', 'dim_date', 'dim_products', 'dim_stores', 'fct_order_items', 'fct_orders', 'int_delivery_timeline', 'int_payments_per_order', 'mart_daily_city_sales', 'qrn_order_items', 'stg_customers', 'stg_delivery_events', 'stg_order_items', 'stg_orders', 'stg_payments', 'stg_products', 'stg_reviews', 'stg_stores', 'stg_supplier_prices']


Every model is listed, including the two placeholders: dbt checks at parse time that every `ref()` points to an existing
model, so we couldn't simply delete them. Let's write them.

## 2 · Write `stg_orders` and build the staging layer (6')

✏️ **Exercise** · Write `models/staging/stg_orders.sql`: one row per order from `source('raw', 'orders')` with the columns
`order_id, customer_id, store_id, status, payment_method, promo_code, delivery_fee`, plus
- `created_at_utc` (cast `created_at` to `timestamp`),
- `created_at_riyadh` and `order_date` (Riyadh date) using the `to_riyadh()` macro,
- `updated_at_utc`.

Write the file from Python (the cell writes `STG_ORDERS` to disk).

In [6]:
STG_ORDERS = """
-- One row per order. The app stores UTC as text; Qafila reports in Riyadh time.
select
    order_id,
    customer_id,
    store_id,
    status,
    payment_method,
    promo_code,
    delivery_fee,
    created_at::timestamp                                as created_at_utc,
    {{ to_riyadh('created_at::timestamp') }}             as created_at_riyadh,
    {{ to_riyadh('created_at::timestamp') }}::date       as order_date,
    updated_at::timestamp                                as updated_at_utc
from {{ source('raw', 'orders') }}
"""
(PROJ / "models/staging/stg_orders.sql").write_text(STG_ORDERS.strip() + "\n", encoding="utf-8")
dbt("run", "--select", "staging")

13:28:41  Running with dbt=1.12.6
13:28:41  Registered adapter: duckdb=1.11.0
13:28:42  Found 19 models, 43 data tests, 1 seed, 9 sources, 506 macros, 3 unit tests
13:28:42  
13:28:42  Concurrency: 4 threads (target='dev')
13:28:42  
13:28:42  2 of 9 START sql view model staging.stg_delivery_events ........................ [RUN]
13:28:42  1 of 9 START sql view model staging.stg_customers .............................. [RUN]
13:28:42  4 of 9 START sql view model staging.stg_orders ................................. [RUN]
13:28:42  3 of 9 START sql view model staging.stg_order_items ............................ [RUN]
13:28:42  1 of 9 OK created sql view model staging.stg_customers ......................... [OK in 0.22s]
13:28:42  2 of 9 OK created sql view model staging.stg_delivery_events ................... [OK in 0.24s]
13:28:42  4 of 9 OK created sql view model staging.stg_orders ............................ [OK in 0.23s]
13:28:42  3 of 9 OK created sql view model staging.stg_order_it

dbtRunnerResult(success=True, exception=None, result=RunExecutionResult(results=[RunResult(status=<RunStatus.Success: 'success'>, timing=[TimingInfo(name='compile', started_at=datetime.datetime(2026, 10, 21, 13, 28, 42, 429694), completed_at=datetime.datetime(2026, 10, 21, 13, 28, 42, 452735)), TimingInfo(name='execute', started_at=datetime.datetime(2026, 10, 21, 13, 28, 42, 488438), completed_at=datetime.datetime(2026, 10, 21, 13, 28, 42, 633030))], thread_id='Thread-3 (worker)', execution_time=0.22420740127563477, adapter_response={'_message': 'OK'}, message='OK', failures=None, node=ModelNode(database='qafila', schema='staging', name='stg_customers', resource_type=<NodeType.Model: 'model'>, package_name='qafila', path='staging/stg_customers.sql', original_file_path='models/staging/stg_customers.sql', unique_id='model.qafila.stg_customers', fqn=['qafila', 'staging', 'stg_customers'], alias='stg_customers', checksum=FileHash(name='sha256', checksum='7c828f635e739e85c301e272b35b20f06ed

In [7]:
sql("SELECT order_id, created_at_utc, created_at_riyadh, order_date FROM staging.stg_orders WHERE order_id = 5000002")

,order_id,created_at_utc,created_at_riyadh,order_date
0,5000002,2026-08-01 09:30:35,2026-08-01 12:30:35,2026-08-01


In [8]:
row = sql("SELECT created_at_utc, created_at_riyadh, order_date FROM staging.stg_orders WHERE order_id = 5000002").iloc[0]
U.check((row.created_at_riyadh - row.created_at_utc).total_seconds() == 3 * 3600, "Riyadh time = UTC + 3h")
U.check(sql("SELECT count(*) AS n FROM staging.stg_orders").n[0] == 45766, "one row per order")

✅ Riyadh time = UTC + 3h
✅ one row per order


Peek at what dbt actually sent to DuckDB: `dbt compile` renders the Jinja into plain SQL under `target/compiled/`.

In [9]:
dbt("compile", "--select", "stg_orders", quiet=True)
print(next((PROJ / "target" / "compiled").rglob("stg_orders.sql")).read_text(encoding="utf-8"))

-- One row per order. The app stores UTC as text; Qafila reports in Riyadh time.
select
    order_id,
    customer_id,
    store_id,
    status,
    payment_method,
    promo_code,
    delivery_fee,
    created_at::timestamp                                as created_at_utc,
    (created_at::timestamp + INTERVAL 3 HOUR)             as created_at_riyadh,
    (created_at::timestamp + INTERVAL 3 HOUR)::date       as order_date,
    updated_at::timestamp                                as updated_at_utc
from "qafila"."raw"."orders"
-- One row per order. The app stores UTC as text; Qafila reports in Riyadh time.
select
    order_id,
    customer_id,
    store_id,
    status,
    payment_method,
    promo_code,
    delivery_fee,
    created_at::timestamp                                as created_at_utc,
    (created_at::timestamp + INTERVAL 3 HOUR)             as created_at_riyadh,
    (created_at::timestamp + INTERVAL 3 HOUR)::date       as order_date,
    updated_at::timestamp               

## 3 · Write the incremental `fct_order_items` and build the marts (7')

The order-line fact is the biggest table, and it grows every day. An **incremental** model processes only new or
changed data on later runs:
- `is_incremental()` is true when the table already exists (and you didn't pass `--full-refresh`);
- `{{ this }}` is the existing table, so `max(...)` over it is your high-water mark;
- `unique_key` + `incremental_strategy='delete+insert'` replaces reprocessed rows instead of duplicating them.

✏️ **Exercise** · Write `models/marts/fct_order_items.sql`, the Lab 2.2 fact in dbt form. Grain: one row per order line
with `quantity > 0`. Config: `materialized='incremental'`, `unique_key='order_line_id'`, `incremental_strategy='delete+insert'`,
`on_schema_change='fail'`. On incremental runs, only take orders with `updated_at_utc` later than **3 days before** the max
`order_updated_at_utc` already loaded (late status changes). Columns, in this order (the YAML contract enforces them):
`order_line_id, order_id, line_no, order_date, customer_id (-1 via var('unknown_key') if not in dim_customers), store_id,
product_id, order_status, payment_method, promo_code, quantity, unit_price, discount, gross_amount, net_amount, order_updated_at_utc`.

In [10]:
FCT = """
{{
    config(
        materialized='incremental',
        unique_key='order_line_id',
        incremental_strategy='delete+insert',
        on_schema_change='fail'
    )
}}

with orders as (
    select * from {{ ref('stg_orders') }}
    {% if is_incremental() %}
    where updated_at_utc > (select max(order_updated_at_utc) - interval 3 day from {{ this }})
    {% endif %}
),

customers as (
    select customer_id from {{ ref('dim_customers') }}
)

select
    i.order_line_id,
    i.order_id,
    i.line_no,
    o.order_date,
    coalesce(c.customer_id, {{ var('unknown_key') }})   as customer_id,
    o.store_id,
    i.product_id,
    o.status                                            as order_status,
    o.payment_method,
    o.promo_code,
    i.quantity,
    i.unit_price,
    i.discount,
    i.gross_amount,
    i.net_amount,
    o.updated_at_utc                                    as order_updated_at_utc
from orders o
join {{ ref('stg_order_items') }} i on i.order_id = o.order_id
left join customers c on c.customer_id = o.customer_id
where i.quantity > 0
"""
(PROJ / "models/marts/fct_order_items.sql").write_text(FCT.strip() + "\n", encoding="utf-8")
dbt("seed", quiet=True)
dbt("run", "--select", "+fct_orders")       # fct_orders and everything it depends on

13:28:45  Running with dbt=1.12.6
13:28:45  Registered adapter: duckdb=1.11.0
13:28:46  Found 19 models, 43 data tests, 1 seed, 9 sources, 506 macros, 3 unit tests
13:28:46  
13:28:46  Concurrency: 4 threads (target='dev')
13:28:46  
13:28:46  4 of 10 START sql view model staging.stg_orders ................................ [RUN]
13:28:46  3 of 10 START sql view model staging.stg_order_items ........................... [RUN]
13:28:46  2 of 10 START sql view model staging.stg_delivery_events ....................... [RUN]
13:28:46  1 of 10 START sql view model staging.stg_customers ............................. [RUN]
13:28:46  3 of 10 OK created sql view model staging.stg_order_items ...................... [OK in 0.17s]
13:28:46  4 of 10 OK created sql view model staging.stg_orders ........................... [OK in 0.18s]
13:28:46  5 of 10 START sql view model staging.stg_payments .............................. [RUN]
13:28:46  2 of 10 OK created sql view model staging.stg_delivery_events

dbtRunnerResult(success=True, exception=None, result=RunExecutionResult(results=[RunResult(status=<RunStatus.Success: 'success'>, timing=[TimingInfo(name='compile', started_at=datetime.datetime(2026, 10, 21, 13, 28, 46, 745258), completed_at=datetime.datetime(2026, 10, 21, 13, 28, 46, 756341)), TimingInfo(name='execute', started_at=datetime.datetime(2026, 10, 21, 13, 28, 46, 757326), completed_at=datetime.datetime(2026, 10, 21, 13, 28, 46, 899263))], thread_id='Thread-26 (worker)', execution_time=0.16940975189208984, adapter_response={'_message': 'OK'}, message='OK', failures=None, node=ModelNode(database='qafila', schema='staging', name='stg_order_items', resource_type=<NodeType.Model: 'model'>, package_name='qafila', path='staging/stg_order_items.sql', original_file_path='models/staging/stg_order_items.sql', unique_id='model.qafila.stg_order_items', fqn=['qafila', 'staging', 'stg_order_items'], alias='stg_order_items', checksum=FileHash(name='sha256', checksum='be19ca6378d9d2c3f5a812

In [11]:
facts = sql("SELECT count(*) AS lines, count(DISTINCT order_id) AS orders, round(sum(net_amount)) AS net_sar FROM marts.fct_order_items")
facts

,lines,orders,net_sar
0,172403,45761,7349442.0


In [12]:
U.check(facts.lines[0] == 172545 - 142, "every order line except the 142 with non-positive quantity")
U.check(sql("SELECT count(*) AS n FROM marts.fct_order_items WHERE customer_id = -1").n[0] > 0, "erased customers mapped to -1")

✅ every order line except the 142 with non-positive quantity
✅ erased customers mapped to -1


## 4 · Change a source row; watch the incremental run (5')

Simulate tomorrow's load: one delivered order is refunded on 1 October, so its `status` and `updated_at` change in the
source. Then run the fact model again, incrementally.

In [13]:
with duckdb.connect(str(DB)) as c:
    c.sql("UPDATE raw.orders SET status = 'refunded', updated_at = '2026-10-01 09:15:00' WHERE order_id = 5000002")
before = sql("SELECT order_status, order_updated_at_utc FROM marts.fct_order_items WHERE order_id = 5000002 LIMIT 1")
dbt("run", "--select", "fct_order_items")
after = sql("SELECT order_status, order_updated_at_utc FROM marts.fct_order_items WHERE order_id = 5000002 LIMIT 1")
display(before, after)

13:28:48  Running with dbt=1.12.6
13:28:48  Registered adapter: duckdb=1.11.0
13:28:48  Found 19 models, 43 data tests, 1 seed, 9 sources, 506 macros, 3 unit tests
13:28:49  
13:28:49  Concurrency: 4 threads (target='dev')
13:28:49  
13:28:49  1 of 1 START sql incremental model marts.fct_order_items ....................... [RUN]
13:28:49  1 of 1 OK created sql incremental model marts.fct_order_items .................. [OK in 0.22s]
13:28:49  
13:28:49  Finished running 1 incremental model in 0 hours 0 minutes and 0.35 seconds (0.35s).
13:28:49  
13:28:49  Completed successfully
13:28:49  
13:28:49  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


,order_status,order_updated_at_utc
0,delivered,2026-08-01 10:08:35


,order_status,order_updated_at_utc
0,refunded,2026-10-01 09:15:00


✏️ **Exercise** · Prove the run was incremental and idempotent:
- `n_processed`: how many orders did the incremental filter select? (Count orders in `staging.stg_orders` with
  `updated_at_utc` later than 3 days before the max `order_updated_at_utc` **before** this run: `2026-09-30 23:59:59`.)
- `dupes`: count of `order_line_id` values that appear more than once in the fact (should be 0).

In [14]:
n_processed = sql("""SELECT count(*) AS n FROM staging.stg_orders
                     WHERE updated_at_utc > TIMESTAMP '2026-09-30 23:59:59' - INTERVAL 3 DAY""").n[0]
dupes = sql("SELECT count(*) AS n FROM (SELECT order_line_id FROM marts.fct_order_items GROUP BY 1 HAVING count(*) > 1)").n[0]
print(f"incremental run reprocessed {n_processed:,} of 45,766 orders | duplicate lines: {dupes}")
U.check(after.order_status[0] == "refunded", "the late change reached the fact table")
U.check(dupes == 0, "delete+insert replaced rows instead of duplicating them")

incremental run reprocessed 2,352 of 45,766 orders | duplicate lines: 0
✅ the late change reached the fact table
✅ delete+insert replaced rows instead of duplicating them


🔍 The look-back window (3 days) is a business decision: it must cover how late changes can arrive. Too short and
late refunds are missed forever; too long and every run reprocesses a lot. `dbt run --full-refresh` rebuilds from scratch
when the logic changes.

## 5 · Read the lineage from the manifest (3')

Every `ref()` and `source()` is an edge in a graph. dbt writes the graph to `target/manifest.json`; `dbt docs generate`
adds column metadata and `dbt docs serve` renders it as a website.

✏️ **Exercise** · From the manifest, list the direct parents (`depends_on.nodes`) of `model.qafila.fct_orders`, and the
**direct** children of `model.qafila.stg_orders` (the manifest's `child_map`). Store them in `parents` and `children` (lists of names).

In [15]:
dbt("parse", quiet=True)
manifest = json.loads((PROJ / "target" / "manifest.json").read_text(encoding="utf-8"))
parents = [n.split(".")[-1] for n in manifest["nodes"]["model.qafila.fct_orders"]["depends_on"]["nodes"]]
children = [n.split(".")[-1] for n in manifest["child_map"]["model.qafila.stg_orders"] if n.startswith("model.")]
print("fct_orders <-", sorted(parents))
print("stg_orders ->", sorted(children))
U.check({"fct_order_items", "stg_orders"} <= set(parents), "fct_orders reads the fact and the staging orders")

fct_orders <- ['dim_customers', 'fct_order_items', 'int_delivery_timeline', 'int_payments_per_order', 'stg_orders']
stg_orders -> ['fct_order_items', 'fct_orders']
✅ fct_orders reads the fact and the staging orders


🔍 This graph is how dbt knows the build order, what `+model` selects, and what breaks downstream when a column
changes. Orchestrators (Lab 4.1) import exactly this graph to schedule each model as an asset.

## ✱ Stretch · Add a weekly category mart and document it

Add `models/marts/mart_weekly_category.sql`: one row per ISO week and product category with delivered revenue and units,
built from `fct_order_items` and `dim_products`. Add a YAML entry with a description and a uniqueness test on
`iso_week || category`, then `dbt build --select mart_weekly_category`.

In [16]:
(PROJ / "models/marts/mart_weekly_category.sql").write_text("""
select
    weekofyear(f.order_date)            as iso_week,
    p.category,
    round(sum(f.net_amount), 2)         as revenue_sar,
    sum(f.quantity)                     as units
from {{ ref('fct_order_items') }} f
join {{ ref('dim_products') }} p using (product_id)
where f.order_status = 'delivered'
group by all
""".strip() + "\n", encoding="utf-8")
(PROJ / "models/marts/_weekly.yml").write_text("""
version: 2
models:
  - name: mart_weekly_category
    description: Delivered revenue and units per ISO week (Riyadh dates) and product category.
    data_tests:
      - unique:
          arguments:
            column_name: "iso_week || '|' || category"
""".strip() + "\n", encoding="utf-8")
dbt("run", "--select", "dim_products", quiet=True)   # not built by +fct_orders
dbt("build", "--select", "mart_weekly_category")
display(sql("SELECT * FROM marts.mart_weekly_category ORDER BY iso_week DESC, revenue_sar DESC LIMIT 5"))

13:28:52  Running with dbt=1.12.6
13:28:52  Registered adapter: duckdb=1.11.0
13:28:52  Found 20 models, 44 data tests, 1 seed, 9 sources, 506 macros, 3 unit tests
13:28:52  
13:28:52  Concurrency: 4 threads (target='dev')
13:28:52  
13:28:52  1 of 2 START sql table model marts.mart_weekly_category ........................ [RUN]
13:28:52  1 of 2 OK created sql table model marts.mart_weekly_category ................... [OK in 0.06s]
13:28:52  2 of 2 START test unique_mart_weekly_category_iso_week_category ................ [RUN]
13:28:52  2 of 2 PASS unique_mart_weekly_category_iso_week_category ...................... [PASS in 0.06s]
13:28:52  
13:28:52  Finished running 1 table model, 1 test in 0 hours 0 minutes and 0.26 seconds (0.26s).
13:28:52  
13:28:52  Completed successfully
13:28:52  
13:28:52  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=2


,iso_week,category,revenue_sar,units
0,40,Dates & Coffee,92913.86,1718.0
1,40,Meat & Poultry,74708.97,1603.0
2,40,Dairy,39230.18,2350.0
3,40,Pantry,25632.23,1580.0
4,40,Fruits & Vegetables,25072.62,2815.0


## Checkpoint

1. Why `{{ source() }}` and `{{ ref() }}` instead of table names? What would break without them?
2. Staging models are views and marts are tables here. When would you make a staging model a table?
3. A bug in `stg_orders` is fixed. Is the incremental `fct_order_items` now correct? What do you run?
4. dbt v2 (the Rust "Fusion" engine) parses projects much faster and understands SQL. What changes for this project? What doesn't?